In [1]:
import os
import json
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
with open("data_split.json", "r") as f:
    split_data = json.load(f)

train_subjects = split_data["train"]
val_subjects = split_data["validation"]
test_subjects = split_data["test"]

print("Train:", len(train_subjects))
print("Validation:", len(val_subjects))
print("Test:", len(test_subjects))

Train: 1000
Validation: 125
Test: 126


In [3]:
train_data = r"Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

In [4]:
def preprocess_t2f(image):
    # Expected original BraTS shape
    if image.shape != (240, 240, 155):
        raise ValueError(f"Unexpected image shape: {image.shape}")

    image = image[16:224, 8:232, :]

    image = np.pad(
        image,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    foreground = image > 0

    if not np.any(foreground):
        raise ValueError("No foreground voxels found")

    upper = np.percentile(image[foreground], 99.9)

    image = np.clip(image, 0, upper)

    image = image / upper

    image[~foreground] = 0

    return image.astype(np.float32)

In [5]:
def preprocess_mask(mask):
    # Expected original BraTS shape
    if mask.shape != (240, 240, 155):
        raise ValueError(f"Unexpected mask shape: {mask.shape}")

    # Apply exactly the same spatial crop as T2f
    mask = mask[16:224, 8:232, :]

    # Apply exactly the same z-padding as T2f
    mask = np.pad(
        mask,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    return mask.astype(np.int64)

In [6]:
def calculate_tumour_entropy(image, mask, num_bins=256):
    # Whole tumour = all non-zero tumour labels
    tumour_region = mask > 0

    if not np.any(tumour_region):
        raise ValueError("No tumour voxels found")

    tumour_values = image[tumour_region]

    # T2f has already been normalized to [0, 1]
    tumour_values = np.clip(tumour_values, 0.0, 1.0)

    hist, _ = np.histogram(
        tumour_values,
        bins=num_bins,
        range=(0.0, 1.0),
        density=False
    )

    probabilities = hist.astype(np.float64)
    probabilities = probabilities / probabilities.sum()

    probabilities = probabilities[probabilities > 0]

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    return np.float32(entropy)

In [7]:
from torch.utils.data import Dataset, DataLoader

class BraTSDataset(Dataset):
    def __init__(self, subjects, data_dir):
        self.subjects = subjects
        self.data_dir = data_dir

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        subject = self.subjects[idx]
        subject_path = os.path.join(self.data_dir, subject)

        files = os.listdir(subject_path)

        t2f_file = [f for f in files if "t2f" in f.lower()][0]
        seg_file = [f for f in files if "seg" in f.lower()][0]

        # Load T2f
        image = nib.load(
            os.path.join(subject_path, t2f_file)
        ).get_fdata()

        # Load segmentation
        mask = nib.load(
            os.path.join(subject_path, seg_file)
        ).get_fdata()

        # Apply preprocessing
        image = preprocess_t2f(image)
        mask = preprocess_mask(mask)

        # Calculate whole-tumour Shannon entropy
        entropy = calculate_tumour_entropy(
            image,
            mask
        )

        # Convert to tensors
        image = torch.from_numpy(
            image
        ).float().unsqueeze(0)

        mask = torch.from_numpy(
            mask
        ).long().unsqueeze(0)

        entropy = torch.tensor(
            entropy,
            dtype=torch.float32
        )

        return {
            "image": image,
            "mask": mask,
            "heterogeneity": entropy,
            "subject": subject
        }

In [8]:
train_dataset = BraTSDataset(
    subjects=train_subjects,
    data_dir=train_data
)

print("Dataset size:", len(train_dataset))

Dataset size: 1000


In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0
)

In [10]:
sample = train_dataset[0]

print("Subject:", sample["subject"])
print("Image:", sample["image"].shape)
print("Mask:", sample["mask"].shape)
print("Mask labels:", torch.unique(sample["mask"]))
print("Heterogeneity:", sample["heterogeneity"])
print("Heterogeneity shape:", sample["heterogeneity"].shape)

Subject: BraTS-GLI-00240-000
Image: torch.Size([1, 208, 224, 160])
Mask: torch.Size([1, 208, 224, 160])
Mask labels: tensor([0, 1, 2, 3])
Heterogeneity: tensor(6.7840)
Heterogeneity shape: torch.Size([])


In [11]:
import numpy as np

entropy_values = []
tumour_volumes = []
subject_ids = []

for i in range(len(train_dataset)):
    sample = train_dataset[i]

    image_np = sample["image"][0].numpy()
    mask_np = sample["mask"][0].numpy()

    entropy = calculate_tumour_entropy(
        image_np,
        mask_np
    )

    # Whole tumour volume in voxels
    tumour_volume = np.sum(mask_np > 0)

    entropy_values.append(entropy)
    tumour_volumes.append(tumour_volume)
    subject_ids.append(sample["subject"])

entropy_values = np.array(entropy_values)
tumour_volumes = np.array(tumour_volumes)

print("Number of subjects:", len(entropy_values))

print("\nEntropy:")
print("Min:", entropy_values.min())
print("Max:", entropy_values.max())
print("Mean:", entropy_values.mean())
print("Median:", np.median(entropy_values))
print("Std:", entropy_values.std())

print("\nPercentiles:")
print("P10:", np.percentile(entropy_values, 10))
print("P25:", np.percentile(entropy_values, 25))
print("P50:", np.percentile(entropy_values, 50))
print("P75:", np.percentile(entropy_values, 75))
print("P90:", np.percentile(entropy_values, 90))

correlation = np.corrcoef(
    entropy_values,
    tumour_volumes
)[0, 1]

print("\nEntropy vs tumour volume correlation:")
print(correlation)

Number of subjects: 1000

Entropy:
Min: 5.57837
Max: 7.7397027
Mean: 6.870206
Median: 6.9228325
Std: 0.3320367

Percentiles:
P10: 6.43379
P25: 6.687603
P50: 6.9228325
P75: 7.1082687
P90: 7.2417426

Entropy vs tumour volume correlation:
0.1639315482471202


In [12]:
class VAEBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels
            ),
            nn.SiLU(),

            nn.Conv3d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),
            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels
            ),
            nn.SiLU()
        )

    def forward(self, x):
        return self.block(x)

In [13]:
class VAEEncoder3D(nn.Module):
    def __init__(
        self,
        in_channels=1,
        base_channels=16,
        latent_channels=4
    ):
        super().__init__()

        self.enc1 = VAEBlock3D(
            in_channels,
            base_channels
        )

        self.down1 = nn.Conv3d(
            base_channels,
            base_channels * 2,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.enc2 = VAEBlock3D(
            base_channels * 2,
            base_channels * 2
        )

        self.down2 = nn.Conv3d(
            base_channels * 2,
            base_channels * 4,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.enc3 = VAEBlock3D(
            base_channels * 4,
            base_channels * 4
        )

        self.down3 = nn.Conv3d(
            base_channels * 4,
            base_channels * 8,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.bottleneck = VAEBlock3D(
            base_channels * 8,
            base_channels * 8
        )

        self.to_mu = nn.Conv3d(
            base_channels * 8,
            latent_channels,
            kernel_size=1
        )

        self.to_logvar = nn.Conv3d(
            base_channels * 8,
            latent_channels,
            kernel_size=1
        )

    def forward(self, x):

        x = self.enc1(x)
        x = self.down1(x)

        x = self.enc2(x)
        x = self.down2(x)

        x = self.enc3(x)
        x = self.down3(x)

        x = self.bottleneck(x)

        mu = self.to_mu(x)
        logvar = self.to_logvar(x)

        return mu, logvar

In [14]:
def reparameterize(mu, logvar):
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return mu + eps * std

In [15]:
class VAEDecoder3D(nn.Module):
    def __init__(
        self,
        out_channels=1,
        base_channels=16,
        latent_channels=4
    ):
        super().__init__()

        self.from_latent = nn.Conv3d(
            latent_channels,
            base_channels * 8,
            kernel_size=3,
            padding=1
        )

        self.dec3 = VAEBlock3D(
            base_channels * 8,
            base_channels * 8
        )

        self.up3 = nn.ConvTranspose3d(
            base_channels * 8,
            base_channels * 4,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.dec2 = VAEBlock3D(
            base_channels * 4,
            base_channels * 4
        )

        self.up2 = nn.ConvTranspose3d(
            base_channels * 4,
            base_channels * 2,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.dec1 = VAEBlock3D(
            base_channels * 2,
            base_channels * 2
        )

        self.up1 = nn.ConvTranspose3d(
            base_channels * 2,
            base_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.final_block = VAEBlock3D(
            base_channels,
            base_channels
        )

        self.output_conv = nn.Conv3d(
            base_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, z):
        x = self.from_latent(z)

        x = self.dec3(x)
        x = self.up3(x)

        x = self.dec2(x)
        x = self.up2(x)

        x = self.dec1(x)
        x = self.up1(x)

        x = self.final_block(x)

        x = self.output_conv(x)

        # T2f preprocessing range = [0, 1]
        x = torch.sigmoid(x)

        return x

In [16]:
class VAE3D(nn.Module):
    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base_channels=16,
        latent_channels=4
    ):
        super().__init__()

        self.encoder = VAEEncoder3D(
            in_channels=in_channels,
            base_channels=base_channels,
            latent_channels=latent_channels
        )

        self.decoder = VAEDecoder3D(
            out_channels=out_channels,
            base_channels=base_channels,
            latent_channels=latent_channels
        )

    def forward(self, x):
        mu, logvar = self.encoder(x)

        z = reparameterize(
            mu,
            logvar
        )

        reconstruction = self.decoder(z)

        return reconstruction, mu, logvar, z

In [17]:
def vae_loss(
    reconstruction,
    target,
    mu,
    logvar,
    kl_weight=1e-6
):
    # Reconstruction loss
    recon_loss = F.l1_loss(
        reconstruction,
        target
    )

    # KL divergence
    kl_loss = -0.5 * torch.mean(
        1
        + logvar
        - mu.pow(2)
        - logvar.exp()
    )

    total_loss = (
        recon_loss
        + kl_weight * kl_loss
    )

    return total_loss, recon_loss, kl_loss

In [18]:
def train_vae(
    model,
    train_loader,
    epochs,
    optimizer,
    device,
    checkpoint_dir="vae_checkpoints",
    kl_weight=1e-6
):
    import os

    os.makedirs(checkpoint_dir, exist_ok=True)

    loss_history = []
    recon_history = []
    kl_history = []

    for epoch in range(epochs):

        model.train()

        epoch_loss = 0.0
        epoch_recon = 0.0
        epoch_kl = 0.0

        for batch_idx, batch in enumerate(train_loader):

            x = batch["image"].to(device)

            optimizer.zero_grad()

            reconstruction, mu, logvar, z = model(x)

            loss, recon_loss, kl_loss = vae_loss(
                reconstruction,
                x,
                mu,
                logvar,
                kl_weight=kl_weight
            )

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_recon += recon_loss.item()
            epoch_kl += kl_loss.item()

            if (batch_idx + 1) % 10 == 0:
                print(
                    f"Epoch {epoch + 1}/{epochs} | "
                    f"Batch {batch_idx + 1}/{len(train_loader)} | "
                    f"Loss: {loss.item():.6f} | "
                    f"Recon: {recon_loss.item():.6f} | "
                    f"KL: {kl_loss.item():.6f}"
                )

        avg_loss = epoch_loss / len(train_loader)
        avg_recon = epoch_recon / len(train_loader)
        avg_kl = epoch_kl / len(train_loader)

        loss_history.append(avg_loss)
        recon_history.append(avg_recon)
        kl_history.append(avg_kl)

        print(
            f"Epoch {epoch + 1} completed | "
            f"Loss: {avg_loss:.6f} | "
            f"Recon: {avg_recon:.6f} | "
            f"KL: {avg_kl:.6f}"
        )

        checkpoint_path = os.path.join(
            checkpoint_dir,
            f"vae_epoch_{epoch + 1:03d}.pt"
        )

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss": avg_loss,
                "recon_loss": avg_recon,
                "kl_loss": avg_kl
            },
            checkpoint_path
        )

        print("Saved:", checkpoint_path)

        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_loss_history.npy"
            ),
            np.array(loss_history)
        )

        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_recon_history.npy"
            ),
            np.array(recon_history)
        )

        np.save(
            os.path.join(
                checkpoint_dir,
                "vae_kl_history.npy"
            ),
            np.array(kl_history)
        )

    return loss_history, recon_history, kl_history

In [19]:
def load_vae_checkpoint(
    model,
    optimizer,
    path,
    device
):
    checkpoint = torch.load(
        path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    if optimizer is not None:
        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

    loaded_epoch = checkpoint["epoch"]

    print(
        f"Loaded VAE checkpoint from epoch {loaded_epoch}"
    )

    return loaded_epoch

In [20]:
@torch.no_grad()
def reconstruct_vae(
    model,
    image,
    device
):
    model.eval()

    image = image.to(device)

    reconstruction, mu, logvar, z = model(image)

    return reconstruction

In [21]:
timesteps = 1000

beta_start = 1e-4
beta_end = 0.02

betas = torch.linspace(beta_start, beta_end, timesteps)

alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

In [22]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2

        embeddings = math.log(10000) / (half_dim - 1)

        embeddings = torch.exp(
            torch.arange(half_dim, device=device) * -embeddings
        )

        embeddings = t[:, None] * embeddings[None, :]

        embeddings = torch.cat(
            (embeddings.sin(), embeddings.cos()),
            dim=1
        )

        return embeddings

In [23]:
class ResBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.conv1 = nn.Conv3d(
            in_channels, out_channels,
            kernel_size=3, padding=1
        )

        self.conv2 = nn.Conv3d(
            out_channels, out_channels,
            kernel_size=3, padding=1
        )

        self.norm1 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels
        )

        self.norm2 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels
        )

        self.time_mlp = nn.Linear(
            time_dim,
            out_channels
        )

        if in_channels != out_channels:
            self.residual = nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        else:
            self.residual = nn.Identity()

    def forward(self, x, t):
        h = self.conv1(x)
        h = self.norm1(h)
        h = F.silu(h)

        time_emb = self.time_mlp(t)
        time_emb = time_emb[:, :, None, None, None]

        h = h + time_emb

        h = self.conv2(h)
        h = self.norm2(h)
        h = F.silu(h)

        return h + self.residual(x)

In [24]:
class DownBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim):
        super().__init__()

        self.resblock = ResBlock3D(
            in_channels,
            out_channels,
            time_dim
        )

        self.downsample = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

    def forward(self, x, t):
        h = self.resblock(x, t)

        down = self.downsample(h)

        return h, down


class UpBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        time_dim
    ):
        super().__init__()

        self.upsample = nn.ConvTranspose3d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.resblock = ResBlock3D(
            out_channels + skip_channels,
            out_channels,
            time_dim
        )

    def forward(self, x, skip, t):
        x = self.upsample(x)

        # Match spatial size to the skip connection.
        # Required because latent dimensions such as 26 -> 13 -> 6
        # cannot be exactly restored by x2 transposed convolution.
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(
                x,
                size=skip.shape[2:],
                mode="trilinear",
                align_corners=False
            )

        x = torch.cat(
            [x, skip],
            dim=1
        )

        x = self.resblock(
            x,
            t
        )

        return x

In [25]:
def prepare_latent_mask(mask, latent_size=(26, 28, 20)):
    """
    Convert BraTS integer mask to 3-channel one-hot mask
    and downsample it to latent spatial resolution.

    Input:
        mask: [B, 1, 208, 224, 160]

    Output:
        latent_mask: [B, 3, 26, 28, 20]
    """

    mask = mask.long().squeeze(1)

    # Tumour classes 1, 2, 3
    mask_onehot = torch.stack(
        [
            (mask == 1),
            (mask == 2),
            (mask == 3)
        ],
        dim=1
    ).float()

    latent_mask = F.interpolate(
        mask_onehot,
        size=latent_size,
        mode="nearest"
    )

    return latent_mask

In [26]:
class ConditionalLatentUNet3D(nn.Module):
    def __init__(
        self,
        latent_channels=4,
        mask_channels=3,
        base_channels=16,
        time_dim=128
    ):
        super().__init__()

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        self.heterogeneity_embedding = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        total_in_channels = latent_channels + mask_channels

        self.input_conv = nn.Conv3d(
            total_in_channels,
            base_channels,
            kernel_size=3,
            padding=1
        )

        self.down1 = DownBlock3D(
            base_channels,
            base_channels * 2,
            time_dim
        )

        self.down2 = DownBlock3D(
            base_channels * 2,
            base_channels * 4,
            time_dim
        )

        self.mid = ResBlock3D(
            base_channels * 4,
            base_channels * 4,
            time_dim
        )

        self.up2 = UpBlock3D(
            in_channels=base_channels * 4,
            skip_channels=base_channels * 4,
            out_channels=base_channels * 2,
            time_dim=time_dim
        )

        self.up1 = UpBlock3D(
            in_channels=base_channels * 2,
            skip_channels=base_channels * 2,
            out_channels=base_channels,
            time_dim=time_dim
        )

        self.output_conv = nn.Conv3d(
            base_channels,
            latent_channels,
            kernel_size=1
        )

    def forward(
        self,
        z,
        t,
        latent_mask,
        heterogeneity
    ):
        # Combine noisy latent + spatial mask condition
        z = torch.cat(
            [z, latent_mask],
            dim=1
        )

        # Timestep embedding
        t_emb = self.time_embedding(t)

        # Heterogeneity embedding
        heterogeneity = heterogeneity.float().view(-1, 1)

        h_emb = self.heterogeneity_embedding(
            heterogeneity
        )

        condition_emb = t_emb + h_emb

        # Latent UNet
        z = self.input_conv(z)

        skip1, z = self.down1(
            z,
            condition_emb
        )

        skip2, z = self.down2(
            z,
            condition_emb
        )

        z = self.mid(
            z,
            condition_emb
        )

        z = self.up2(
            z,
            skip2,
            condition_emb
        )

        z = self.up1(
            z,
            skip1,
            condition_emb
        )

        z = self.output_conv(z)

        return z

In [27]:
def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)

    device = x0.device

    sqrt_alpha_cumprod_device = sqrt_alphas_cumprod.to(device)
    sqrt_one_minus_alpha_cumprod_device = (
        sqrt_one_minus_alphas_cumprod.to(device)
    )

    sqrt_alpha_hat = (
        sqrt_alpha_cumprod_device[t]
        .view(-1, 1, 1, 1, 1)
    )

    sqrt_one_minus_alpha_hat = (
        sqrt_one_minus_alpha_cumprod_device[t]
        .view(-1, 1, 1, 1, 1)
    )

    xt = (
        sqrt_alpha_hat * x0
        + sqrt_one_minus_alpha_hat * noise
    )

    return xt, noise

In [28]:
def save_checkpoint(model, optimizer, epoch, path):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict()
    }, path)


def load_checkpoint(model, optimizer, path, device):
    checkpoint = torch.load(path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])

    if optimizer is not None:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    return checkpoint["epoch"]

In [29]:
device = torch.device("cuda")

vae = VAE3D(
    in_channels=1,
    out_channels=1,
    base_channels=16,
    latent_channels=4
).to(device)

optimizer = torch.optim.Adam(
    vae.parameters(),
    lr=1e-4
)

train_vae(
    model=vae,
    train_loader=train_loader,
    epochs=10,
    optimizer=optimizer,
    device=device,
    checkpoint_dir="vae_checkpoints",
    kl_weight=1e-6
)

Epoch 1/10 | Batch 10/1000 | Loss: 0.351839 | Recon: 0.351838 | KL: 0.247315


Epoch 1/10 | Batch 20/1000 | Loss: 0.332101 | Recon: 0.332100 | KL: 0.398765


Epoch 1/10 | Batch 30/1000 | Loss: 0.311827 | Recon: 0.311826 | KL: 0.502952


Epoch 1/10 | Batch 40/1000 | Loss: 0.295062 | Recon: 0.295061 | KL: 0.538263


Epoch 1/10 | Batch 50/1000 | Loss: 0.279585 | Recon: 0.279585 | KL: 0.540950


...

Epoch 10/10 | Batch 970/1000 | Loss: 0.010690 | Recon: 0.010686 | KL: 4.600688


Epoch 10/10 | Batch 980/1000 | Loss: 0.009229 | Recon: 0.009225 | KL: 4.084959


Epoch 10/10 | Batch 990/1000 | Loss: 0.007575 | Recon: 0.007571 | KL: 4.020949


Epoch 10/10 | Batch 1000/1000 | Loss: 0.012258 | Recon: 0.012254 | KL: 4.604242
Epoch 10 completed | Loss: 0.010947 | Recon: 0.010943 | KL: 4.191689
Saved: vae_checkpoints/vae_epoch_010.pt


([0.18938863553851842,
  0.0730329286083579,
  0.036218722127377985,
  0.02235424330830574,
  0.0167196791311726,
  0.014002045786008239,
  0.01233492804551497,
  0.011910725564695895,
  0.01125320776551962,
  0.010946700476575642],
 [0.1893876455873251,
  0.07303116148710251,
  0.036216598596423864,
  0.022351812660694123,
  0.016716869123280048,
  0.013998936779331415,
  0.012331528120208532,
  0.011907094198279082,
  0.011249268940184265,
  0.01094250877154991],
 [0.9902404380440712,
  1.7670906846523284,
  2.123499933958054,
  2.430652730703354,
  2.8100409986972807,
  3.1090171930789947,
  3.399959547281265,
  3.631378862142563,
  3.938825037240982,
  4.191689251899719])